# 01 — Provision Lakebase + reverse-sync gold tables

What this notebook does:
1. Creates a Lakebase Postgres instance named `maas-team8` (CU_1)
2. Registers the `acme_agency` database as a UC database catalog
3. Adds primary key constraints to the UC source tables (synced tables require these)
4. Creates synced tables for each gold table (SNAPSHOT scheduling — refresh on demand)
5. Creates and seeds `tenant_membership` (email -> advertiser_id mapping)

Idempotent — re-runs are safe.

In [ ]:
import os, time, uuid, json
from databricks.sdk import WorkspaceClient

INSTANCE = os.environ.get('LAKEBASE_INSTANCE', 'maas-team8')
LB_DB    = os.environ.get('LAKEBASE_DATABASE', 'acme_agency')
CATALOG  = os.environ.get('CATALOG', 'alexander_booth')
SCHEMA   = os.environ.get('SCHEMA',  'maas_summit_team8')

w = WorkspaceClient()
print(f'instance={INSTANCE}  db={LB_DB}  source={CATALOG}.{SCHEMA}')

## 1. Create Lakebase instance (idempotent)

In [ ]:
def get_instance():
    try:
        return w.api_client.do('GET', f'/api/2.0/database/instances/{INSTANCE}')
    except Exception:
        return None

inst = get_instance()
if inst is None:
    print(f'creating {INSTANCE}...')
    inst = w.api_client.do('POST', '/api/2.0/database/instances', body={'name': INSTANCE, 'capacity': 'CU_1'})
    while inst['state'] not in ('AVAILABLE', 'FAILED'):
        time.sleep(15)
        inst = get_instance()
        print(f"  state={inst['state']}")
print(f"{INSTANCE}: state={inst['state']}  host={inst['read_write_dns']}")

## 2. Register the postgres database as a UC database catalog

In [ ]:
try:
    cat = w.catalogs.get(LB_DB)
    print(f'catalog {LB_DB} exists: options={getattr(cat,"options",None)}')
except Exception:
    print(f'creating catalog {LB_DB}...')
    w.api_client.do('POST', '/api/2.0/database/catalogs', body={
        'name': LB_DB,
        'database_instance_name': INSTANCE,
        'database_name': LB_DB,
        'create_database_if_not_exists': True,
    })
    print('done')

## 3. Add primary keys to UC source tables

In [ ]:
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

constraints = [
    ('advertisers',        ['advertiser_id']),
    ('campaigns',          ['campaign_id']),
    ('campaign_daily_perf',['advertiser_id','campaign_id','date']),
    ('audience_segments',  ['segment_id']),
    ('pacing_metrics',     ['campaign_id']),
]
for table, pks in constraints:
    fq = f'{CATALOG}.{SCHEMA}.{table}'
    for c in pks:
        spark.sql(f'ALTER TABLE {fq} ALTER COLUMN {c} SET NOT NULL')
    spark.sql(f'ALTER TABLE {fq} DROP CONSTRAINT IF EXISTS {table}_pk')
    spark.sql(f'ALTER TABLE {fq} ADD CONSTRAINT {table}_pk PRIMARY KEY ({", ".join(pks)})')
    print(f'  PK on {fq}: ({", ".join(pks)})')

## 4. Create synced tables (UC gold -> Lakebase Postgres)

In [ ]:
for table, pks in constraints:
    name = f'{LB_DB}.public.{table}'
    try:
        w.api_client.do('GET', f'/api/2.0/database/synced_tables/{name}')
        print(f'  sync exists: {name}')
        continue
    except Exception:
        pass
    body = {
        'name': name,
        'database_instance_name': INSTANCE,
        'logical_database_name': LB_DB,
        'spec': {
            'source_table_full_name': f'{CATALOG}.{SCHEMA}.{table}',
            'primary_key_columns': pks,
            'scheduling_policy': 'SNAPSHOT',
            'create_database_objects_if_missing': True,
            'new_pipeline_spec': {'storage_catalog': CATALOG, 'storage_schema': SCHEMA},
        },
    }
    r = w.api_client.do('POST', '/api/2.0/database/synced_tables', body=body)
    print(f"  created sync {name}: {r.get('data_synchronization_status',{}).get('detailed_state')}")

In [ ]:
# Poll until all syncs are ONLINE
while True:
    states = []
    for table, _ in constraints:
        r = w.api_client.do('GET', f'/api/2.0/database/synced_tables/{LB_DB}.public.{table}')
        s = r.get('data_synchronization_status', {}).get('detailed_state','?')
        states.append((table, s))
    print(' '.join(f'{t}={s}' for t,s in states))
    if all('ONLINE' in s for _, s in states):
        print('all ONLINE'); break
    time.sleep(20)

## 5. Create + seed tenant_membership

Map login emails to advertiser_id. Adjust to your team's emails before showing the demo.

In [ ]:
import asyncio, asyncpg
me_email = w.current_user.me().user_name
inst = w.api_client.do('GET', f'/api/2.0/database/instances/{INSTANCE}')
host = inst['read_write_dns']
cred = w.api_client.do('POST', '/api/2.0/database/credentials',
                       body={'instance_names':[INSTANCE], 'request_id': str(uuid.uuid4())})

SEED = [
    ('alexander.booth@databricks.com', 'adv_patagonia',    'agency_admin'),
    ('joe.hu@databricks.com',         'adv_patagonia',    'agency_admin'),
    ('megan.tupper@databricks.com',   'adv_patagonia',    'agency_admin'),
    ('michelle.lui@databricks.com',   'adv_patagonia',    'agency_admin'),
    ('media@patagonia.com',           'adv_patagonia',    'viewer'),
    ('digital@allbirds.com',          'adv_allbirds',     'viewer'),
    ('paidmedia@bombas.com',          'adv_bombas',       'viewer'),
    ('analytics@warbyparker.com',     'adv_warbyparker',  'viewer'),
    ('growth@glossier.com',           'adv_glossier',     'viewer'),
    ('marketing@outdoorvoices.com',   'adv_outdoorvoices','viewer'),
    ('founders@caraway.com',          'adv_caraway',      'viewer'),
    ('brand@brightland.com',          'adv_brightland',   'viewer'),
    ('cmo@olipop.com',                'adv_olipop',       'viewer'),
]

async def go():
    conn = await asyncpg.connect(host=host, port=5432, user=me_email,
                                  password=cred['token'], database=LB_DB, ssl='require')
    await conn.execute('''
        CREATE TABLE IF NOT EXISTS public.tenant_membership (
            email TEXT NOT NULL, advertiser_id TEXT NOT NULL, role TEXT NOT NULL DEFAULT 'viewer',
            PRIMARY KEY (email, advertiser_id)
        )''')
    await conn.executemany(
        'INSERT INTO public.tenant_membership(email,advertiser_id,role) VALUES($1,$2,$3) ON CONFLICT DO NOTHING',
        SEED,
    )
    n = await conn.fetchval('SELECT COUNT(*) FROM public.tenant_membership')
    print(f'tenant_membership: {n} rows')
    # smoke test
    spend = await conn.fetchval(
        'SELECT SUM(spend_usd) FROM public.campaign_daily_perf WHERE advertiser_id=$1', 'adv_patagonia')
    print(f'patagonia 12w spend = ${spend:,.0f}')
    await conn.close()
asyncio.run(go())